<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


## Leveraging Apache Spark for Smart Building HVAC Monitoring

**Estimated time needed: 30 minutes**

### Objectives

After completing this lab, you will be able to:

- Explain the distributed architecture of Spark in the context of smart building monitoring
- Simulate real-time sensor data for HVAC systems in a building
- Perform SQL queries to detect critical environmental conditions and calculate average readings
- Determine the aggregated results to the console for immediate insights into room conditions


## Background
Smart Building Solutions, Inc. specializes in optimizing HVAC (heating, ventilation, and air conditioning) systems to enhance comfort and energy efficiency in commercial buildings. By monitoring temperature and humidity levels in real-time across various rooms, the company aims to ensure optimal indoor conditions and preemptively address potential HVAC issues.

With a continuous influx of sensor data, Smart Building Solutions needs to process and analyze this data in real-time to maintain the quality of the indoor environment.

## Data set description
The simulated data set comprises:

`room_id`: Unique identifier for each room (e.g., R001, R002).

`temperature`: Current temperature reading from the sensor (in °C).

`humidity`: Current humidity level reading from the sensor (in %).

`timestamp`: Time when the reading was recorded (automatically generated by Spark).
The data is generated at a rate of 5 rows per second, simulating multiple rooms with various environmental conditions.


## Challenges
Monitoring indoor environmental conditions poses several challenges:

**High data velocity**: Continuous data from multiple sensors can overwhelm traditional systems.

**Need for immediate alerts**: Delays in identifying critical conditions can lead to discomfort or system inefficiencies.

**Need for data aggregation and analysis**: Efficiently aggregating and analyzing real-time data for proactive maintenance and optimization is essential.

## Apache Spark with structured streaming
To address these challenges, Apache Spark is employed for its powerful distributed computing capabilities, enabling real-time data processing and analytics.


In [1]:
!pip install pyspark==3.1.2 -q
!pip install findspark -q

In [2]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# FindSpark simplifies the process of using Apache Spark with Python

import findspark
findspark.init()

#import functions/Classes for sparkml

from pyspark.ml.clustering import KMeans


from pyspark.sql import SparkSession


### Set up the Spark session:


In [3]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Smart Building HVAC Monitoring") \
    .getOrCreate()


26/04/22 11:18:32 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Simulate sensor data:

Use Spark’s rate source to generate continuous readings from multiple rooms.


In [4]:
from pyspark.sql.functions import expr, rand,when

# Simulate sensor data with room IDs and readings
sensor_data = spark.readStream.format("rate").option("rowsPerSecond", 5).load() \
    .withColumn("room_id", expr("CAST(value % 10 AS STRING)")) \
    .withColumn("temperature", when(expr("value % 10 == 0"), 15)  # Set temperature to 15 for one specific record
                .otherwise(20 + rand() * 25)) \
    .withColumn("humidity", expr("40 + rand() * 30"))

### Create a temporary SQL view:

Create temporary SQL view to perform SQL queries on the streaming data.


In [5]:
# Create a temporary SQL view for the sensor data
sensor_data.createOrReplaceTempView("sensor_table")


### Define SQL queries for aggregation and analysis:

* **Critical temperature query**: Detect rooms with critical temperature levels
* **Average readings query**: Calculate average readings over a 1-minute window
* **Attention needed query**: Identify rooms that need immediate attention based on humidity levels


In [6]:
# SQL Query to detect rooms with critical temperatures
critical_temperature_query = """
    SELECT 
        room_id, 
        temperature, 
        humidity, 
        timestamp 
    FROM sensor_table 
    WHERE temperature < 18 OR temperature > 60
"""

# SQL Query to calculate average readings over a 1-minute window
average_readings_query = """
    SELECT 
        room_id, 
        AVG(temperature) AS avg_temperature, 
        AVG(humidity) AS avg_humidity, 
        window.start AS window_start 
    FROM sensor_table
    GROUP BY room_id, window(timestamp, '1 minute')
"""

# SQL Query to find rooms that need immediate attention based on humidity
attention_needed_query = """
    SELECT 
        room_id, 
        COUNT(*) AS critical_readings 
    FROM sensor_table 
    WHERE humidity < 45 OR humidity > 75
    GROUP BY room_id
"""


### Execute the SQL queries:

Execute each SQL query to create streaming DataFrames.


In [7]:
# Execute the critical temperature query
critical_temperatures_stream = spark.sql(critical_temperature_query)

# Execute the average readings query
average_readings_stream = spark.sql(average_readings_query)

# Execute the attention needed query
attention_needed_stream = spark.sql(attention_needed_query)


### Output the results to the console:

Display the results from each query in real-time.


In [8]:
# Output the results to the console for all queries
critical_query = critical_temperatures_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .queryName("Critical Temperatures") \
    .start()

average_query = average_readings_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Average Readings") \
    .start()

attention_query = attention_needed_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Attention Needed") \
    .start()



### Keep the streams running:

Ensure that the streaming queries continue to run to process incoming data.


In [9]:
# Keep the streams running

print("********Critical Temperature Values*******")
critical_query.awaitTermination(60)
print("********Average Readings Values********")
average_query.awaitTermination(60)
print("********Attention Needed Values********")
attention_query.awaitTermination(60)


********Critical Temperature Values*******
-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 59.01194714435055|2026-04-22 11:18:...|
|      0|       15.0| 68.45465068473791|2026-04-22 11:18:...|
|      0|       15.0|46.407429857387044|2026-04-22 11:18:...|
|      0|       15.0| 45.14598084544875|2026-04-22 11:18:...|
+-------+-----------+------------------+--------------------+



-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
+-------+-----------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|50.968429109993835|2026-04-22 11:19:...|
|      0|       15.0|48.303026877409394|2026-04-22 11:19:...|
|      0|       15.0| 48.72611364389169|2026-04-22 11:18:...|
|      0|       15.0| 47.31510550731939|2026-04-22 11:18:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+---------------+------------+------------+
|room_id|avg_temperature|avg_humidity|window_start|
+-------+---------------+------------+------------+
+-------+---------------+------------+------------+

********Average Readings Values********


-------------------------------------------
Batch: 3
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|59.895570701373885|2026-04-22 11:19:...|
|      0|       15.0|  65.7829231862822|2026-04-22 11:19:...|
|      0|       15.0| 53.82890676291403|2026-04-22 11:19:...|
|      0|       15.0|  49.8720614823951|2026-04-22 11:19:...|
|      0|       15.0| 69.51221506201678|2026-04-22 11:19:...|
|      0|       15.0| 59.04412484326017|2026-04-22 11:19:...|
|      0|       15.0| 46.50388045889103|2026-04-22 11:19:...|
|      0|       15.0| 57.69681293511682|2026-04-22 11:19:...|
|      0|       15.0|59.423597397009715|2026-04-22 11:19:...|
|      0|       15.0| 41.34264638637676|2026-04-22 11:19:...|
|      0|       15.0| 43.01488801912302|2026-04-22 11:19:...|
|      0|       15.0| 69.0484599300

...

-------------------------------------------
Batch: 2
-------------------------------------------
+-------+------------------+------------------+-------------------+
|room_id|   avg_temperature|      avg_humidity|       window_start|
+-------+------------------+------------------+-------------------+
|      3|33.027102636739535| 52.95991284521422|2026-04-22 11:18:00|
|      8|33.487422969089806|53.672107269440204|2026-04-22 11:19:00|
|      2|32.684129931197624|  49.0369147400947|2026-04-22 11:18:00|
|      3|34.760395144994156| 55.30783815808719|2026-04-22 11:20:00|
|      6|31.044264162604417| 57.28335258740827|2026-04-22 11:20:00|
|      3| 32.04953067278021| 55.60847666444493|2026-04-22 11:19:00|
|      4|29.983012822554528|  54.4180464765465|2026-04-22 11:19:00|
|      9| 33.23037640156328| 56.60173680375256|2026-04-22 11:20:00|
|      5| 31.79526720011635|53.809890685602234|2026-04-22 11:19:00|
|      7|34.462400560356606| 61.05621101460952|2026-04-22 11:18:00|
|      7| 31.748481

-------------------------------------------
Batch: 7
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|59.098240331526185|2026-04-22 11:20:...|
|      0|       15.0| 66.09680702562976|2026-04-22 11:20:...|
|      0|       15.0| 62.79618901580356|2026-04-22 11:20:...|
|      0|       15.0| 41.87354135279983|2026-04-22 11:20:...|
|      0|       15.0| 69.69311463451804|2026-04-22 11:20:...|
|      0|       15.0| 48.94451663370885|2026-04-22 11:20:...|
|      0|       15.0| 40.09562279878746|2026-04-22 11:20:...|
|      0|       15.0| 65.30719480700799|2026-04-22 11:20:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
+-------+-----------------+
|room_id|critical_readings|
+----

-------------------------------------------
Batch: 8
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0|52.309118262599334|2026-04-22 11:21:...|
|      0|       15.0| 57.31499328074625|2026-04-22 11:21:...|
|      0|       15.0| 69.02594130565147|2026-04-22 11:21:...|
|      0|       15.0|48.787385848724746|2026-04-22 11:21:...|
|      0|       15.0|55.844788207517134|2026-04-22 11:21:...|
|      0|       15.0|60.723714512432764|2026-04-22 11:21:...|
|      0|       15.0| 57.40818680281813|2026-04-22 11:21:...|
|      0|       15.0| 61.74079212647129|2026-04-22 11:21:...|
|      0|       15.0|60.242190886467164|2026-04-22 11:21:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 3
-------------------------------------------

-------------------------------------------
Batch: 9
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0|       15.0| 47.21605722980204|2026-04-22 11:21:...|
|      0|       15.0|53.535963213642376|2026-04-22 11:21:...|
|      0|       15.0|  51.4991726629966|2026-04-22 11:21:...|
|      0|       15.0| 68.87550075560887|2026-04-22 11:21:...|
|      0|       15.0|  64.6095262648681|2026-04-22 11:21:...|
|      0|       15.0|  65.7614327738062|2026-04-22 11:21:...|
|      0|       15.0|62.608201036732254|2026-04-22 11:21:...|
|      0|       15.0| 44.73362831640873|2026-04-22 11:21:...|
+-------+-----------+------------------+--------------------+

-------------------------------------------
Batch: 4
-------------------------------------------


False

In [10]:
spark.streams.active

+-------+-----------------+
|room_id|critical_readings|
+-------+-----------------+
|      7|               14|
|      3|               12|
|      8|               11|
|      0|                6|
|      5|                7|
|      6|               13|
|      9|               11|
|      1|                8|
|      4|               11|
|      2|                9|
+-------+-----------------+



In [ ]:
critical_query.stop()
average_query.stop()
attention_query.stop()

In [ ]:
spark.stop()

### Conclusion
In this lab, you explored the use of Apache Spark in smart building monitoring, particularly focusing on HVAC (heating, ventilation, and air conditioning) systems. You now understand the Spark's distributed architecture. You also understand how to simulate real-time sensor data for temperature and humidity, execute SQL queries to identify critical environmental conditions, and output aggregated results for immediate insights.


## Author(s)

Lakshmi Holla

## Other Contributors
Malika Singla
